# Quark: train a model locally

Trains a Quark model on this machine's GPU with a live dashboard, best/latest checkpoints, early stopping and automatic resume.

**Before running:** select the project's `.venv` as the notebook kernel (VS Code: *Select Kernel → Python Environments → .venv*). If it isn't there, run `uv sync --extra dev` in the repo first.

Edit only the **Settings** cell, then *Run All*. Re-running the notebook resumes the latest run automatically.

## 1. Environment check

In [ ]:
import sys
import torch
import quark
from quark.hardware import detect_profile, load_profile

print(f"python {sys.version.split()[0]} | torch {torch.__version__} (CUDA {torch.version.cuda}) | quark {quark.__version__}")
assert torch.cuda.is_available(), "No GPU visible to PyTorch." + (
    " Runtime -> Change runtime type -> T4 GPU." if "google.colab" in sys.modules
    else " Is the notebook using the project's .venv kernel (CUDA build of torch)?")

hw = detect_profile()
print(f"GPU: {hw.gpu_name} | {hw.vram_gb:g} GB ({hw.usable_vram_gb:g} usable) | precision {hw.precision} | SDPA {hw.sdpa_kernels}")
TARGET = "colab-t4" if "google.colab" in sys.modules else "rtx3070ti-laptop"
if hw.compute_capability != load_profile(TARGET).compute_capability:
    print(f"warning: this GPU differs from the expected target ({TARGET}); settings may need adjusting")

## Settings
The only cell you're expected to edit.

In [ ]:
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()  # the quark repo
MODELS_ROOT = ROOT / "models"
DATA_ROOT = ROOT / "data"

MODEL_NAME = "ts-small"                  # folder under models/
PRESET = "tinystories-small"             # used only when the model folder doesn't exist yet
DATA_SOURCE = "roneneldan/TinyStories"   # Hugging Face dataset, or a local file/folder
TOKENIZER = "train:8192"                 # train:<vocab> or pretrained:<name> (plan §7.4 rule)
RESUME = "auto"                          # auto | new | latest | best | path to a .pt file
MAX_STEPS = None                         # e.g. 200 for a quick test; None = full run

## Model config

In [ ]:
from quark.config import TokenizerConfig
from quark.config.presets import get_preset
from quark.project import create_model, load_model_config, model_dir

MODEL_DIR = model_dir(MODEL_NAME, root=MODELS_ROOT)
if not (MODEL_DIR / "config.yaml").exists():
    create_model(MODEL_NAME, get_preset(PRESET), root=MODELS_ROOT)
    print(f"created {MODEL_DIR / 'config.yaml'} from preset {PRESET!r}")
cfg = load_model_config(MODEL_DIR)

kind, _, value = TOKENIZER.partition(":")
tok_cfg = TokenizerConfig(type="train", size=int(value)) if kind == "train" else TokenizerConfig(type="pretrained", name=value, size=None)
cfg = cfg.model_copy(update={"data": cfg.data.model_copy(update={"source": DATA_SOURCE, "tokenizer": tok_cfg})})
m, t = cfg.model, cfg.train
print(f"{cfg.name}: d_model {m.d_model}, {m.n_layers} layers, {m.n_heads} heads, seq_len {m.seq_len}, vocab {m.vocab_size}")
print(f"train: lr {t.lr}, {t.tokens_per_step(m.seq_len):,} tokens/step, {t.max_tokens / 1e6:.0f}M tokens, precision {t.precision}")

## Data
Tokenized once, then reused. The first run downloads the dataset and trains the tokenizer (several minutes for TinyStories).

In [ ]:
from quark.data import TokenData, default_data_dir, prepare_dataset

DATA_DIR = default_data_dir(cfg.data, root=DATA_ROOT)
if not (DATA_DIR / "meta.json").exists():
    print(f"preparing {DATA_SOURCE} -> {DATA_DIR} ...")
    prepare_dataset(cfg.data, DATA_DIR, progress=lambda d, n: d % 100_000 == 0 and print(f"  {d:,} docs, {n / 1e6:,.0f}M tokens"))

In [ ]:
from quark.data import MixData, load_data

data = load_data(DATA_DIR)
meta = data.meta
if isinstance(data, MixData):
    weights = data.resolve_weights(cfg.data.weights)
    print(f"mix {meta['name']} | vocab {meta['vocab_size']:,}")
    for name, src in meta["sources"].items():
        w = weights.get(name, 0.0)
        epochs = w * cfg.train.max_tokens / max(1, src["train_tokens"])
        print(f"  {name:18s} train {src['train_tokens'] / 1e6:7,.1f}M | val {src['val_tokens'] / 1e6:5,.2f}M | "
              f"weight {w:4.0%} | {epochs:.2f} epochs planned")
    sample_tokens = next(iter(data.sources.values())).val
else:
    print(f"vocab {meta['vocab_size']:,} | train {meta['train_tokens'] / 1e6:,.1f}M tokens ({meta['train_docs']:,} docs) "
          f"| val {meta['val_tokens'] / 1e6:,.2f}M tokens ({meta['val_docs']:,} docs)")
    epochs = cfg.train.max_tokens / meta["train_tokens"]
    print(f"planned training: {epochs:.2f} epochs" + ("  (multi-epoch: watch for overfitting)" if epochs > 1 else ""))
    sample_tokens = data.val
print("\nsample (decoded from val):\n" + data.tokenizer().decode(sample_tokens[:150].tolist()))

## Model and memory check
One real training step on the GPU: the actual peak memory, not an estimate.

In [ ]:
from quark.train import dry_run

check = dry_run(cfg, DATA_DIR)
print(f"{check['params_m']:.2f}M params | {check['precision']} | peak GPU memory {check['peak_mem_gb']:.2f} GB "
      f"of {hw.usable_vram_gb:g} usable | {check['tokens_per_sec'] / 1e3:.1f}k tokens/s "
      f"| full run ~{check['est_hours']:.1f} h")
if check["peak_mem_gb"] > hw.usable_vram_gb:
    print("warning: this config won't fit; lower train.micro_batch and raise train.grad_accum in config.yaml")

## Resume detection
`RESUME = "auto"` continues the latest run if it has a checkpoint; otherwise a new run starts.

In [ ]:
from quark.project import plan_run

RUN_DIR, CHECKPOINT = plan_run(MODEL_DIR, RESUME)
print(("resuming " + str(CHECKPOINT)) if CHECKPOINT else ("new run: " + str(RUN_DIR)))

## Train
Live dashboard below; stats print every few steps. Interrupting the cell (stop button) saves `latest.pt` first.

In [ ]:
from quark.train import Trainer

trainer = Trainer(cfg, DATA_DIR, RUN_DIR, hardware=hw, resume_from=CHECKPOINT, live_plot=True, max_steps=MAX_STEPS)
result = trainer.fit()

## Results

In [ ]:
from IPython.display import Image, display

print(f"stopped: {result.stop_reason} | steps {result.steps:,} | tokens {result.tokens / 1e6:,.1f}M | "
      f"best val loss {result.best_val:.4f} | last val loss {result.last_val:.4f}")
print(f"run folder: {result.run_dir}")
display(Image(filename=str(result.run_dir / "training.png")))

## Samples from the best checkpoint

In [ ]:
from quark.infer import generate_text, load_for_inference

model, tokenizer, _ = load_for_inference(result.run_dir / "best.pt")
for i, prompt in enumerate(["Once upon a time", "The little dog", "Lily wanted to"]):
    print(f"--- {prompt!r} ---")
    print(generate_text(model, tokenizer, prompt, max_new_tokens=200, temperature=0.8, top_p=0.9, seed=i), "\n")

## Export
Exporting to Hugging Face / safetensors format comes in Phase 6 (`quark export`). Checkpoints in the run folder already work with `quark generate` and `load_for_inference`.